# J1S2 — Pandas & NumPy
**Formation Data Science · Jour 1 · Session 2 · 10h30 – 12h00**

> **Prérequis :** J1S1 complétée — repo forké, CSV dans `data/raw/`, PR J1S1 mergée  
> **Livrable :** `data/processed/credit_features_j1.parquet` (16 colonnes)  
> **Dataset :** `credit_risk_dataset.csv` (32 581 lignes · 12 colonnes · CC0)

---

## 🎯 Objectifs

- **Inspecter** le DataFrame : dtypes, describe, isnull, nunique
- **Agréger** : groupby, agg(), pivot_table — taux de défaut par segment
- **Filtrer** : boolean indexing, loc/iloc, query()
- **Créer des features** : loan_to_income, credit_risk_score, age_bucket, income_quartile
- **Vectoriser** avec NumPy : np.where, np.percentile, np.corrcoef
- **Sauvegarder** en Parquet : `data/processed/credit_features_j1.parquet`


---

## ⚙️ Cellule d'initialisation — Exécuter en premier

Cette cellule définit `ROOT` (racine du projet) et vérifie l'environnement.  
Elle fonctionne en **VS Code local** et en **Google Colab**.


In [4]:
from pathlib import Path
import os, sys, subprocess

# ── Racine du projet ─────────────────────────────────────────────────────────
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

# Fallback Colab : si lancé depuis la racine du repo
if not (ROOT / "requirements.txt").exists() and (Path.cwd() / "requirements.txt").exists():
    ROOT = Path.cwd()

os.chdir(ROOT)
print(f"Racine du projet : {ROOT}")

# ── Installation des dépendances depuis requirements.txt ─────────────────────
# Cette cellule est autonome : elle installe tout ce qu'il faut sans dépendre de J1S1
req_path = ROOT / "requirements.txt"
if req_path.exists():
    print("\nInstallation des dépendances (requirements.txt)...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(req_path),
         "-q", "--no-warn-script-location"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅ Dépendances installées")
    else:
        print("⚠️  Erreur installation :")
        print(result.stderr[-500:])
else:
    # Fallback si requirements.txt absent (ex. notebook ouvert hors du repo)
    print("\n⚠️  requirements.txt introuvable — installation minimale pour J1S2...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "pandas", "numpy", "pyarrow", "-q"],
        capture_output=True
    )
    print("✅ pandas, numpy, pyarrow installés")

# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import pyarrow  # requis pour to_parquet / read_parquet (Bloc 6)

# ── Versions ─────────────────────────────────────────────────────────────────
print(f"\nPython  : {sys.version.split()[0]}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"PyArrow : {pyarrow.__version__}")

# ── Détection de l'environnement ─────────────────────────────────────────────
if ".venv" in sys.executable:
    print("\n✅ Virtualenv .venv actif (VS Code)")
elif "google.colab" in sys.modules:
    print("\n✅ Google Colab détecté")
else:
    print(f"\n⚠️  Environnement : {sys.executable}")

# ── Vérification du CSV ───────────────────────────────────────────────────────
csv_path = ROOT / "data" / "raw" / "credit_risk_dataset.csv"
if csv_path.exists():
    print(f"\n✅ Dataset trouvé : {csv_path.name}")
else:
    print(f"\n❌ Dataset introuvable dans data/raw/")
    print("   → Copier credit_risk_dataset.csv dans data/raw/ avant de continuer")


Racine du projet : d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation

Installation des dépendances (requirements.txt)...
✅ Dépendances installées

Python  : 3.10.11
Pandas  : 2.3.3
NumPy   : 2.2.6
PyArrow : 24.0.0

✅ Virtualenv .venv actif (VS Code)

✅ Dataset trouvé : credit_risk_dataset.csv


---

## Bloc 1 — Inspection complète du DataFrame

**Objectif :** lire le dataset et comprendre sa structure avant toute manipulation.

> 🔑 Règle : **toujours inspecter avant de transformer**. On ne modifie pas ce qu'on ne comprend pas.


In [5]:
# ── Chargement du dataset ────────────────────────────────────────────────
df = pd.read_csv(ROOT / "data" / "raw" / "credit_risk_dataset.csv")

print(f"Shape : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print()
print("Colonnes et types :")
print(df.dtypes)


Shape : 32,581 lignes × 12 colonnes

Colonnes et types :
person_age                      int64
person_income                   int64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length      int64
dtype: object


In [6]:
# ── Statistiques descriptives ─────────────────────────────────────────────
df.describe(include="all").round(2)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
count,32581.00,32581.00,32581,31686.00,32581,32581,32581.00,29465.00,32581.00,32581.00,32581,32581.00
unique,NaN,NaN,4,NaN,6,7,NaN,NaN,NaN,NaN,2,NaN
top,NaN,NaN,RENT,NaN,EDUCATION,A,NaN,NaN,NaN,NaN,N,NaN
freq,NaN,NaN,16446,NaN,6453,10777,NaN,NaN,NaN,NaN,26836,NaN
mean,27.73,66074.85,NaN,4.79,NaN,NaN,9589.37,11.01,0.22,0.17,NaN,5.80
std,6.35,61983.12,NaN,4.14,NaN,NaN,6322.09,3.24,0.41,0.11,NaN,4.06
min,20.00,4000.00,NaN,0.00,NaN,NaN,500.00,5.42,0.00,0.00,NaN,2.00
25%,23.00,38500.00,NaN,2.00,NaN,NaN,5000.00,7.90,0.00,0.09,NaN,3.00
50%,26.00,55000.00,NaN,4.00,NaN,NaN,8000.00,10.99,0.00,0.15,NaN,4.00
75%,30.00,79200.00,NaN,7.00,NaN,NaN,12200.00,13.47,0.00,0.23,NaN,8.00


In [7]:
# ── Valeurs manquantes ───────────────────────────────────────────────────
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)

missing = pd.DataFrame({
    "nb_manquants": nulls,
    "pct": nulls_pct
}).query("nb_manquants > 0")

print("Colonnes avec valeurs manquantes :")
print(missing)
print()
print("⚠️  Ces NaN seront traités en J1S4 (imputation médiane)")


Colonnes avec valeurs manquantes :
                   nb_manquants  pct
person_emp_length           895  2.7
loan_int_rate              3116  9.6

⚠️  Ces NaN seront traités en J1S4 (imputation médiane)


In [8]:
# ── Cardinalité et distribution des variables catégorielles ──────────────
print("Cardinalité par colonne :")
print(df.nunique())
print()

# Distribution des variables catégorielles
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col} :")
    print(df[col].value_counts())


Cardinalité par colonne :
person_age                      58
person_income                 4295
person_home_ownership            4
person_emp_length               36
loan_intent                      6
loan_grade                       7
loan_amnt                      753
loan_int_rate                  348
loan_status                      2
loan_percent_income             77
cb_person_default_on_file        2
cb_person_cred_hist_length      29
dtype: int64


person_home_ownership :
person_home_ownership
RENT        16446
MORTGAGE    13444
OWN          2584
OTHER         107
Name: count, dtype: int64

loan_intent :
loan_intent
EDUCATION            6453
MEDICAL              6071
VENTURE              5719
PERSONAL             5521
DEBTCONSOLIDATION    5212
HOMEIMPROVEMENT      3605
Name: count, dtype: int64

loan_grade :
loan_grade
A    10777
B    10451
C     6458
D     3626
E      964
F      241
G       64
Name: count, dtype: int64

cb_person_default_on_file :
cb_person_default_on_file
N  

In [9]:
# ── Taux de défaut — chiffre de référence de toute la formation ──────────
taux_defaut = df["loan_status"].mean()
nb_defauts  = df["loan_status"].sum()

print(f"Taux de défaut global : {taux_defaut:.1%}")
print(f"Dossiers en défaut    : {nb_defauts:,} / {len(df):,}")
print()
print("✅ Vérification : taux doit être 22.0 %")


Taux de défaut global : 21.8%
Dossiers en défaut    : 7,108 / 32,581

✅ Vérification : taux doit être 22.0 %


---

## Bloc 2 — Groupby & agrégations métier

**Objectif :** calculer les taux de défaut par segment — la base de tout reporting crédit.

> 💡 Le résultat de cette cellule est directement utilisable par un comité de risque réel.


In [10]:
# ── Taux de défaut par grade — groupby simple ────────────────────────────
risk_by_grade = (
    df.groupby("loan_grade")["loan_status"]
    .agg(taux_defaut="mean", nb_prets="count")
    .round(3)
    .sort_values("taux_defaut", ascending=False)
)

print("Taux de défaut par grade de risque :")
print(risk_by_grade)
print()
print("📌 Grades F & G : taux > 50 % → dossiers à refuser ou taux fortement majoré")


Taux de défaut par grade de risque :
            taux_defaut  nb_prets
loan_grade                       
G                 0.984        64
F                 0.705       241
E                 0.644       964
D                 0.590      3626
C                 0.207      6458
B                 0.163     10451
A                 0.100     10777

📌 Grades F & G : taux > 50 % → dossiers à refuser ou taux fortement majoré


In [11]:
# ── Agrégations multiples simultanées ───────────────────────────────────
stats_par_grade = df.groupby("loan_grade").agg(
    taux_defaut    = ("loan_status",        "mean"),
    revenu_median  = ("person_income",       "median"),
    montant_moyen  = ("loan_amnt",           "mean"),
    taux_interet   = ("loan_int_rate",       "mean"),
    nb_dossiers    = ("loan_status",         "count"),
).round(2)

stats_par_grade.sort_values("taux_defaut", ascending=False)


,taux_defaut,revenu_median,montant_moyen,taux_interet,nb_dossiers
loan_grade,,,,,
G,0.98,66900.0,17195.70,20.25,64
F,0.71,65000.0,14717.32,18.61,241
E,0.64,59328.5,12915.85,17.01,964
D,0.59,53888.0,10849.24,15.36,3626
C,0.21,53000.0,9213.86,13.46,6458
B,0.16,55000.0,9995.48,11.00,10451
A,0.10,57600.0,8539.27,7.33,10777


In [12]:
# ── Pivot table : vue croisée grade × type de logement ──────────────────
pivot = pd.pivot_table(
    df,
    values="loan_status",
    index="loan_grade",
    columns="person_home_ownership",
    aggfunc="mean",
).round(3)

print("Taux de défaut : grade × type de logement")
pivot


Taux de défaut : grade × type de logement


person_home_ownership,MORTGAGE,OTHER,OWN,RENT
loan_grade,,,,
A,0.044,0.115,0.071,0.175
B,0.080,0.147,0.042,0.241
C,0.142,0.353,0.063,0.266
D,0.457,0.550,0.064,0.738
E,0.475,0.750,0.544,0.755
F,0.629,1.000,0.533,0.780
G,1.000,NaN,1.000,0.964


In [13]:
# ── Taux de défaut par tranche de revenu (qcut) ──────────────────────────
df_temp = df.copy()
df_temp["income_bin"] = pd.qcut(df["person_income"], q=4,
                                 labels=["Q1","Q2","Q3","Q4"])

defaut_par_revenu = (
    df_temp.groupby("income_bin", observed=True)["loan_status"]
    .agg(taux_defaut="mean", nb_dossiers="count")
    .round(3)
)

print("Taux de défaut par quartile de revenu :")
print(defaut_par_revenu)


Taux de défaut par quartile de revenu :
            taux_defaut  nb_dossiers
income_bin                          
Q1                0.397         8152
Q2                0.213         8231
Q3                0.171         8055
Q4                0.091         8143


---

## Bloc 3 — Filtrage & sélection avancée

**Objectif :** extraire des sous-ensembles métier — dossiers haut risque, profils spécifiques.

| Méthode | Usage |
|---------|-------|
| `df[mask]` | Boolean indexing — filtre ligne à ligne |
| `df.loc[mask, cols]` | Sélection par **labels** |
| `df.iloc[rows, cols]` | Sélection par **positions entières** |
| `df.query()` | Syntaxe SQL-like — lisible en reporting |


In [14]:
# ── Boolean indexing — dossiers haut risque ──────────────────────────────
mask_haut_risque = (
    df["loan_grade"].isin(["F", "G"]) &
    (df["loan_percent_income"] > 0.3)
)

dossiers_haut_risque = df[mask_haut_risque]
print(f"Dossiers haut risque : {len(dossiers_haut_risque):,}")
print(f"Taux de défaut dans ce segment : {dossiers_haut_risque['loan_status'].mean():.1%}")


Dossiers haut risque : 68
Taux de défaut dans ce segment : 88.2%


In [15]:
# ── loc[] : sélection par labels ─────────────────────────────────────────
# Colonnes financières pour les dossiers en défaut
cols_financiers = ["person_income", "loan_amnt", "loan_int_rate",
                   "loan_percent_income", "loan_grade"]

defauts = df.loc[df["loan_status"] == 1, cols_financiers]
print(f"Dossiers en défaut : {len(defauts):,}")
defauts.head(5)


Dossiers en défaut : 7,108


,person_income,loan_amnt,loan_int_rate,loan_percent_income,loan_grade
0,59000,35000,16.02,0.59,D
2,9600,5500,12.87,0.57,C
3,65500,35000,15.23,0.53,C
4,54400,35000,14.27,0.55,C
5,9900,2500,7.14,0.25,A


In [16]:
# ── iloc[] : sélection par positions ─────────────────────────────────────
print("5 premières lignes, 4 premières colonnes :")
print(df.iloc[0:5, 0:4])
print()
print("Dernières 3 lignes, colonnes 0 et 4 :")
print(df.iloc[-3:, [0, 4]])


5 premières lignes, 4 premières colonnes :
   person_age  person_income person_home_ownership  person_emp_length
0          22          59000                  RENT              123.0
1          21           9600                   OWN                5.0
2          25           9600              MORTGAGE                1.0
3          23          65500                  RENT                4.0
4          24          54400                  RENT                8.0

Dernières 3 lignes, colonnes 0 et 4 :
       person_age      loan_intent
32578          65  HOMEIMPROVEMENT
32579          56         PERSONAL
32580          66          MEDICAL


In [17]:
# ── query() : syntaxe SQL-like ───────────────────────────────────────────
haut_risque_q = df.query(
    "loan_grade in ['E','F','G'] and "
    "loan_percent_income > 0.25 and "
    "loan_status == 1"
)

print(f"Dossiers défaut grade E/F/G avec ratio > 25 % : {len(haut_risque_q):,}")
print()
# Profil moyen de ce segment
print("Profil moyen :")
print(haut_risque_q[["person_income","loan_amnt","loan_int_rate"]].mean().round(0))


Dossiers défaut grade E/F/G avec ratio > 25 % : 342

Profil moyen :
person_income    52587.0
loan_amnt        18060.0
loan_int_rate       18.0
dtype: float64


---

## Bloc 4 — Feature engineering

**Objectif :** créer 4 nouvelles variables métier qui enrichiront le modèle de scoring (J2S2).

> ⚠️ On travaille directement sur `df` — les nouvelles colonnes seront sauvegardées dans le Parquet.

| Feature créée | Méthode | Rôle |
|---------------|---------|------|
| `loan_to_income` | Division simple | Ratio endettement corrigé |
| `credit_risk_score` | Combinaison pondérée | Score heuristique 0–125 |
| `age_bucket` | `pd.cut()` | Tranche d'âge — intervalles métier |
| `income_quartile` | `pd.qcut()` | Quartile revenu — distribution statistique |


In [18]:
# ── Feature 1 : Ratio prêt / revenu ─────────────────────────────────────
df["loan_to_income"] = (df["loan_amnt"] / df["person_income"]).round(4)

print("loan_to_income — statistiques :")
print(df["loan_to_income"].describe().round(3))
print()
# Corrélation avec le défaut
corr = df[["loan_to_income", "loan_status"]].corr()
print(f"Corrélation avec loan_status : {corr.loc['loan_to_income','loan_status']:.3f}")


loan_to_income — statistiques :
count    32581.000
mean         0.171
std          0.107
min          0.001
25%          0.090
50%          0.148
75%          0.229
max          0.830
Name: loan_to_income, dtype: float64

Corrélation avec loan_status : 0.386


In [19]:
# ── Feature 2 : Score de risque composite (heuristique métier) ───────────
# Imputation temporaire du taux d'intérêt pour le score
loan_int_rate_filled = df["loan_int_rate"].fillna(df["loan_int_rate"].median())  # médiane = 10,99 %

grade_map = {"A": 0, "B": 5, "C": 15, "D": 25, "E": 40, "F": 55, "G": 70}

df["credit_risk_score"] = (
    (df["loan_percent_income"] * 40) +
    (loan_int_rate_filled * 3) +
    (df["loan_grade"].map(grade_map))
).round(1)

print("credit_risk_score — statistiques :")
print(df["credit_risk_score"].describe().round(1))
print()
# Score moyen par statut
print("Score moyen par statut (0=sain, 1=défaut) :")
print(df.groupby("loan_status")["credit_risk_score"].mean().round(1))


credit_risk_score — statistiques :
count    32581.0
mean        48.9
std         20.3
min         16.7
25%         33.0
50%         44.9
75%         61.1
max        150.5
Name: credit_risk_score, dtype: float64

Score moyen par statut (0=sain, 1=défaut) :
loan_status
0    44.3
1    65.5
Name: credit_risk_score, dtype: float64


In [20]:
# ── Feature 3 : Tranche d'âge avec pd.cut() ──────────────────────────────
# cut() = intervalles FIXES définis par la logique métier
df["age_bucket"] = pd.cut(
    df["person_age"],
    bins=[17, 25, 35, 45, 60, 200],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

print("Distribution par tranche d'âge :")
print(df["age_bucket"].value_counts().sort_index())
print()
# Taux de défaut par tranche
print("Taux de défaut par tranche d'âge :")
print(
    df.groupby("age_bucket", observed=True)["loan_status"]
    .mean().round(3).sort_index()
)


Distribution par tranche d'âge :
age_bucket
18-25    15352
26-35    13763
36-45     2814
46-60      582
60+         70
Name: count, dtype: int64

Taux de défaut par tranche d'âge :
age_bucket
18-25    0.230
26-35    0.207
36-45    0.207
46-60    0.218
60+      0.243
Name: loan_status, dtype: float64


In [21]:
# ── Feature 4 : Quartile de revenu avec pd.qcut() ────────────────────────
# qcut() = intervalles DYNAMIQUES basés sur la distribution (équirépartition)
df["income_quartile"] = pd.qcut(
    df["person_income"],
    q=4,
    labels=["Q1_bas", "Q2_moyen_bas", "Q3_moyen_haut", "Q4_haut"]
)

print("Distribution par quartile de revenu :")
print(df["income_quartile"].value_counts().sort_index())
print()
# Taux de défaut par quartile
print("Taux de défaut par quartile de revenu :")
print(
    df.groupby("income_quartile", observed=True)["loan_status"]
    .mean().round(3).sort_index()
)


Distribution par quartile de revenu :
income_quartile
Q1_bas           8152
Q2_moyen_bas     8231
Q3_moyen_haut    8055
Q4_haut          8143
Name: count, dtype: int64

Taux de défaut par quartile de revenu :
income_quartile
Q1_bas           0.397
Q2_moyen_bas     0.213
Q3_moyen_haut    0.171
Q4_haut          0.091
Name: loan_status, dtype: float64


In [22]:
# ── Vérification : shape après feature engineering ───────────────────────
print(f"Shape avant features : (32581, 12)")
print(f"Shape après features : {df.shape}")
print()
print("Nouvelles colonnes :")
new_cols = ["loan_to_income", "credit_risk_score", "age_bucket", "income_quartile"]
print(df[new_cols].dtypes)
print()
df[new_cols].head(5)


Shape avant features : (32581, 12)
Shape après features : (32581, 16)

Nouvelles colonnes :
loan_to_income        float64
credit_risk_score     float64
age_bucket           category
income_quartile      category
dtype: object



,loan_to_income,credit_risk_score,age_bucket,income_quartile
0,0.5932,96.7,18-25,Q3_moyen_haut
1,0.1042,42.4,18-25,Q1_bas
2,0.5729,76.4,18-25,Q1_bas
3,0.5344,81.9,18-25,Q3_moyen_haut
4,0.6434,79.8,18-25,Q2_moyen_bas


---

## Bloc 5 — NumPy vectorisé

**Objectif :** utiliser NumPy pour des opérations rapides — éviter les boucles Python sur 32 581 lignes.

> ⚡ Règle : **ne jamais utiliser `iterrows()` sur un DataFrame de taille réelle**. NumPy est ×50 plus rapide.


In [23]:
import numpy as np

# ── Démonstration : iterrows vs np.where ─────────────────────────────────
import time

# ❌ Version lente (boucle Python)
t0 = time.time()
flag_lent = []
for _, row in df.iterrows():
    flag_lent.append(1 if row["loan_int_rate"] > 15 else 0)
t_lent = time.time() - t0

# ✅ Version rapide (NumPy vectorisé)
t0 = time.time()
df["taux_eleve"] = np.where(df["loan_int_rate"] > 15, 1, 0)
t_rapide = time.time() - t0

print(f"iterrows()  : {t_lent:.2f}s")
print(f"np.where()  : {t_rapide:.4f}s")
print(f"Facteur     : ×{t_lent/max(t_rapide,0.001):.0f} plus rapide")


iterrows()  : 1.58s
np.where()  : 0.0135s
Facteur     : ×118 plus rapide


In [24]:
# ── Statistiques avec NumPy ───────────────────────────────────────────────
revenus = df["person_income"].values  # → ndarray NumPy

p25 = np.percentile(revenus, 25)
p75 = np.percentile(revenus, 75)
iqr = p75 - p25

print("Statistiques des revenus :")
print(f"  Médiane : {np.median(revenus):>12,.0f} $")  # → 55 000 $
print(f"  P25     : {p25:>12,.0f} $")           # → 38 500 $
print(f"  P75     : {p75:>12,.0f} $")           # → 79 200 $
print(f"  IQR     : {iqr:>12,.0f} $")           # → 40 700 $
print(f"  Skewness: {float(np.mean((revenus - np.mean(revenus))**3) / np.std(revenus)**3):>12.2f}")


Statistiques des revenus :
  Médiane :       55,000 $
  P25     :       38,500 $
  P75     :       79,200 $
  IQR     :       40,700 $
  Skewness:        32.86


In [25]:
# ── Matrice de corrélations avec NumPy ───────────────────────────────────
cols_corr = ["loan_amnt", "loan_int_rate", "loan_percent_income",
             "loan_to_income", "credit_risk_score", "loan_status"]

data_corr = df[cols_corr].dropna().values
corr_matrix = np.corrcoef(data_corr.T)

print("Corrélations avec loan_status (défaut) :")
print("-" * 45)
for i, col in enumerate(cols_corr[:-1]):
    bar = "█" * int(abs(corr_matrix[i, -1]) * 20)
    sign = "+" if corr_matrix[i, -1] > 0 else ""
    print(f"  {col:<25} {sign}{corr_matrix[i, -1]:.3f}  {bar}")

print()
print("📌 loan_int_rate et loan_percent_income sont les plus prédictifs du défaut")


Corrélations avec loan_status (défaut) :
---------------------------------------------
  loan_amnt                 +0.107  ██
  loan_int_rate             +0.335  ██████
  loan_percent_income       +0.379  ███████
  loan_to_income            +0.386  ███████
  credit_risk_score         +0.434  ████████

📌 loan_int_rate et loan_percent_income sont les plus prédictifs du défaut


In [26]:
# ── np.clip : borner les outliers d'âge ──────────────────────────────────
# person_age max = 144 (outlier confirmé — probablement erreur de saisie)
# person_income max = 6 000 000 (outlier massif)
# person_emp_length max = 123 (outlier)
print(f"Âge max avant clip : {df['person_age'].max()}")  # → 144

df["person_age_clean"] = np.clip(df["person_age"], 18, 80)

print(f"Âge max après clip : {df['person_age_clean'].max()}")  # → 80 (borné)
print()
print("Distribution après nettoyage :")
print(df["person_age_clean"].describe().round(1))


Âge max avant clip : 144
Âge max après clip : 80

Distribution après nettoyage :
count    32581.0
mean        27.7
std          6.2
min         20.0
25%         23.0
50%         26.0
75%         30.0
max         80.0
Name: person_age_clean, dtype: float64


---

## Bloc 6 — Sauvegarde Parquet & validation

**Objectif :** sauvegarder le DataFrame enrichi en Parquet — le livrable de cette session.

> 📦 **Pourquoi Parquet ?**  
> - Types préservés (catégories, floats, ints)  
> - ~5× plus compact que CSV  
> - Compatible Spark, Arrow, DuckDB (utilisé en J3)  
> - Lecture columnar : lecture partielle rapide  

> ⚠️ Le fichier Parquet est dans `data/processed/` qui est dans le `.gitignore`.  
> Seul le `.gitkeep` est commité — les données restent locales.


In [27]:
# ── Sauvegarder le DataFrame enrichi en Parquet ──────────────────────────
output_path = ROOT / "data" / "processed" / "credit_features_j1.parquet"

df.to_parquet(output_path, index=False)
print(f"✅ Sauvegardé : {output_path}")
print(f"   Taille : {output_path.stat().st_size / 1024:.0f} KB")


✅ Sauvegardé : d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\data\processed\credit_features_j1.parquet
   Taille : 447 KB


In [28]:
# ── Validation immédiate — relire le Parquet ─────────────────────────────
df_check = pd.read_parquet(output_path)

print(f"Shape    : {df_check.shape}")
print(f"Lignes   : {df_check.shape[0]:,}  (attendu : 32 581)")
print(f"Colonnes : {df_check.shape[1]}    (attendu : 16)")
print()
print("Colonnes et types :")
print(df_check.dtypes)


Shape    : (32581, 18)
Lignes   : 32,581  (attendu : 32 581)
Colonnes : 18    (attendu : 16)

Colonnes et types :
person_age                       int64
person_income                    int64
person_home_ownership           object
person_emp_length              float64
loan_intent                     object
loan_grade                      object
loan_amnt                        int64
loan_int_rate                  float64
loan_status                      int64
loan_percent_income            float64
cb_person_default_on_file       object
cb_person_cred_hist_length       int64
loan_to_income                 float64
credit_risk_score              float64
age_bucket                    category
income_quartile               category
taux_eleve                       int64
person_age_clean                 int64
dtype: object


In [29]:
# ── Vérification des features créées ─────────────────────────────────────
new_cols = ["loan_to_income", "credit_risk_score", "age_bucket",
            "income_quartile", "taux_eleve", "person_age_clean"]

print("Nouvelles colonnes :")
print(df_check[new_cols].describe(include="all").round(2))


Nouvelles colonnes :
        loan_to_income  credit_risk_score age_bucket income_quartile  \
count         32581.00           32581.00      32581           32581   
unique             NaN                NaN          5               4   
top                NaN                NaN      18-25    Q2_moyen_bas   
freq               NaN                NaN      15352            8231   
mean              0.17              48.93        NaN             NaN   
std               0.11              20.27        NaN             NaN   
min               0.00              16.70        NaN             NaN   
25%               0.09              33.00        NaN             NaN   
50%               0.15              44.90        NaN             NaN   
75%               0.23              61.10        NaN             NaN   
max               0.83             150.50        NaN             NaN   

        taux_eleve  person_age_clean  
count     32581.00          32581.00  
unique         NaN               NaN

In [30]:
# ── Checklist finale ─────────────────────────────────────────────────────
checks = {
    "32 581 lignes"         : df_check.shape[0] == 32581,
    "16 colonnes minimum"   : df_check.shape[1] >= 16,  # 12 orig + 4 features
    "loan_to_income présent": "loan_to_income" in df_check.columns,
    "credit_risk_score"     : "credit_risk_score" in df_check.columns,
    "age_bucket (catégorie)": str(df_check["age_bucket"].dtype) == "category",
    "Parquet lisible"       : True,
}

print("\n✅ Checklist J1S2 :")
all_ok = True
for label, ok in checks.items():
    status = "✅" if ok else "❌"
    print(f"  {status} {label}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("🎉 Livrable J1S2 validé — prêt pour J1S3 (EDA & Plotly)")
else:
    print("⚠️  Corriger les points marqués ❌ avant de continuer")



✅ Checklist J1S2 :
  ✅ 32 581 lignes
  ✅ 16 colonnes minimum
  ✅ loan_to_income présent
  ✅ credit_risk_score
  ✅ age_bucket (catégorie)
  ✅ Parquet lisible

🎉 Livrable J1S2 validé — prêt pour J1S3 (EDA & Plotly)


---

## Bloc 7 — Commit & push

> ⚠️ Le Parquet lui-même **ne sera pas commité** (gitignore).  
> On commite le notebook mis à jour.

**Décommenter et exécuter les 3 lignes ci-dessous :**


In [ ]:
import subprocess

def git_run(cmd):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=str(ROOT))
    if r.stdout.strip(): print(r.stdout)
    if r.stderr.strip(): print(r.stderr)
    return r.returncode

# ── Décommenter pour exécuter ────────────────────────────────────────────
# git_run("git add notebooks/J1S2_pandas_numpy.ipynb")
# git_run('git commit -m "J1S2: Pandas & NumPy — feature engineering complet"')
# git_run("git push origin main")


---

## ➡️ Prochaine session : J1S3 — EDA & Plotly

**Ce qui nous attend :**
- On repart de `credit_features_j1.parquet` — plus de CSV
- `pd.read_parquet(ROOT / "data/processed/credit_features_j1.parquet")`
- Histogrammes, box plots, scatter matrix avec **Plotly Express**
- Dashboard interactif : taux de défaut par grade, distribution des revenus
- Livrable J1S3 : notebook EDA complet

**Prérequis J1S3 :**
- Avoir le Parquet dans `data/processed/` ✅ (si Bloc 6 exécuté)
- Commit J1S2 pushé sur GitHub
